In [1]:
#
# This cell should fix the missing headers problem
#
import os
import subprocess
from pathlib import Path

ROOT_DIR = Path("/Volumes/fine_tuning/fine_tuning/vol_finetuning")
DEVEL_ROOT = Path(ROOT_DIR) / "python311-devel"
PAYLOAD_DIR = DEVEL_ROOT / "payload"
HEADER_DIR = PAYLOAD_DIR / "usr" / "include" / "python3.11"
python_h = HEADER_DIR / "Python.h"
rpm_file = DEVEL_ROOT / "python3.11-devel.rpm"

if not rpm_file.exists():
    raise RuntimeError(f"RPM non trovato: {rpm_file}")

if not python_h.exists():
    PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

    rpm2cpio = subprocess.Popen(
        ["rpm2cpio", str(rpm_file)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    cpio_result = subprocess.run(
        ["cpio", "-id", "--quiet", "--no-preserve-owner"],
        stdin=rpm2cpio.stdout,
        cwd=PAYLOAD_DIR,
        text=True,
        capture_output=True,
    )

    if rpm2cpio.stdout is not None:
        rpm2cpio.stdout.close()

    rpm_stderr = rpm2cpio.stderr.read().decode() if rpm2cpio.stderr else ""
    rpm_returncode = rpm2cpio.wait()

    if rpm_returncode != 0 or cpio_result.returncode != 0:
        raise RuntimeError(
            "RPM extraction failed.\n"
            f"rpm2cpio: {rpm_stderr}\n"
            f"cpio: {cpio_result.stderr}"
        )

if not python_h.exists():
    raise RuntimeError(f"Python.h not found after extraction: {python_h}")

old_c_include = os.environ.get("C_INCLUDE_PATH", "")
old_cpath = os.environ.get("CPATH", "")

os.environ["C_INCLUDE_PATH"] = (
    f"{HEADER_DIR}:{old_c_include}" if old_c_include else str(HEADER_DIR)
)
os.environ["CPATH"] = (
    f"{HEADER_DIR}:{old_cpath}" if old_cpath else str(HEADER_DIR)
)

print(f"Header available in: {HEADER_DIR}")

Header available in: /Volumes/fine_tuning/fine_tuning/vol_finetuning/python311-devel/payload/usr/include/python3.11
